# 1. Introduction

This project uses the **turnover dataset**, which is situated in the **human resources / workforce analytics** domain. The purpose of the dataset is to predict whether an employee is likely to leave the organization. This type of predictive modeling is important because employee turnover affects recruitment costs, training costs, productivity, and organizational stability. By identifying the factors associated with employee departure, organizations can make better decisions about retention strategies and workforce planning.

In this dataset, the outcome variable is **leave**, which is binary and indicates whether an employee leaves the organization. Therefore, this is a **categorical prediction problem**. The remaining variables are potential predictors describing employee compensation, job perceptions, work conditions, organizational context, and interaction effects. A few derived interaction variables are also included and can be considered predictors. No obvious ID variable appears in the dataset, so at this stage no identifier is excluded in the introduction section.

The table below summarizes the variables, their measurement types, and their roles in the analysis.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/turnover_data.csv")

# Create variable summary table
variable_table = pd.DataFrame({
    "Variable": df.columns,
    "Measurement Type": [
        "Numeric",   # Salary
        "Numeric",   # JobInsecurity
        "Ordinal/Numeric",   # JobSatisfaction
        "Numeric",   # Tenure
        "Numeric",   # EmotionalExhaustion
        "Categorical",   # Domain
        "Categorical",   # OrgLevel
        "Categorical",   # Demographics
        "Ordinal/Numeric",   # SupervisorSatisfaction
        "Ordinal/Numeric",   # CareerAdvancement
        "Ordinal/Numeric",   # WorkLifeBalance
        "Ordinal/Numeric",   # OrgAlignment
        "Ordinal/Numeric",   # LaborMarketCondition
        "Numeric",   # Salary_JS
        "Numeric",   # JS_SupS
        "Numeric",   # Tenure_CA
        "Numeric",   # Exhaustion_WLB
        "Numeric",   # CA_Align
        "Categorical",   # Domain_LMC
        "Binary Outcome"   # leave
    ],
    "Role": [
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Predictor",
        "Outcome"
    ]
})

variable_table

The dataset contains **20 variables** in total. Of these, **19 variables** are candidate predictors and **1 variable** (`leave`) is the outcome. The predictors include a mix of numeric, ordinal, and categorical variables. Several predictors appear to be interaction or engineered variables, such as `Salary_JS`, `JS_SupS`, `Tenure_CA`, `Exhaustion_WLB`, `CA_Align`, and `Domain_LMC`. These may help capture combined effects between employee characteristics and workplace conditions.

The outcome variable in this dataset is **leave**, which represents employee turnover. This variable is binary: a value of **1 indicates that the employee leaves the organization**, while a value of **0 indicates that the employee remains with the organization**. Therefore, the modeling task is a **binary classification problem**, where the objective is to predict the probability that an employee will leave based on the set of predictor variables describing compensation, job perceptions, organizational context, and employee experience.

#2. Exploratory Analysis

This section examines the structure of the dataset and performs the preprocessing required before fitting predictive models. The exploratory analysis includes inspecting variable types, identifying missing values, and preparing the data for modeling. Preprocessing steps include converting categorical variables to the appropriate format, checking for variables that should be excluded from modeling, and evaluating whether centering and scaling are necessary.

Because the dataset includes a mix of numeric and categorical variables, categorical predictors must be converted into factor (category) variables so that machine learning models can properly interpret them. Additionally, some models such as Support Vector Machines, LASSO regression, and neural networks are sensitive to the scale of predictor variables. For these models, centering and scaling are useful to ensure that predictors are measured on comparable scales.

This section therefore performs the following preprocessing steps:

- inspection of dataset structure and summary statistics  
- conversion of categorical variables to category (factor) type  
- verification that the outcome variable is binary  
- preparation of predictors for modeling  
- discussion of centering and scaling decisions

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("/content/turnover_data.csv")

# View first rows
df.head()

The first few rows of the dataset were inspected to understand the structure of the data and confirm that the dataset loaded correctly.

In [ ]:
# Convert categorical variables
categorical_vars = [
    "Domain",
    "OrgLevel",
    "Demographics",
    "Domain_LMC"
]

for col in categorical_vars:
    df[col] = df[col].astype("category")

df.dtypes

In [ ]:
df.info()

The dataset structure shows that the variables consist of both numeric and categorical attributes. Some variables represent employee perceptions or organizational characteristics that should be treated as categorical predictors. These variables are therefore converted to categorical (factor) type so that modeling algorithms interpret them correctly.

In [ ]:
# Convert categorical variables
categorical_vars = [
    "Domain",
    "OrgLevel",
    "Demographics",
    "Domain_LMC"
]

for col in categorical_vars:
    df[col] = df[col].astype("category")

df.dtypes

The variables **Domain**, **OrgLevel**, **Demographics**, and **Domain_LMC** represent categorical organizational characteristics. These variables were converted to categorical data types so that modeling algorithms treat them as discrete groups rather than numeric quantities.

In [ ]:
df["leave"].value_counts()

The outcome variable **leave** represents employee turnover and is a binary variable. A value of **1 indicates that the employee leaves the organization**, while **0 indicates that the employee stays**. Because the outcome variable is binary, the predictive modeling task is a **classification problem**.

In [ ]:
df.isnull().sum()

The dataset was examined for missing values to determine whether imputation or removal of observations would be necessary before model training.

No missing values were detected, so no imputation procedures were required.

In [ ]:
df.nunique()

Variables were inspected to determine whether any predictors should be excluded from model fitting. Variables with extremely low variability or identifier variables would normally be removed because they do not contribute predictive information. In this dataset, no identifier variable is present and all predictors show variability, so no variables were excluded at this stage.

### Centering and Scaling

Centering and scaling were considered because some machine learning algorithms are sensitive to the scale of predictor variables. Methods such as **LASSO regression, Support Vector Machines, and neural networks** rely on distance-based or gradient-based optimization and therefore benefit from standardized predictors.

Tree-based models such as **Random Forests and Gradient Boosted Trees** do not require scaling because they split variables based on thresholds rather than distances.

For this reason, predictor variables will be **standardized when fitting models that require scaling**, while tree-based models will be fit using the original predictor scales.

# 3. Building Predictive Models

This section develops several predictive models to estimate the probability that an employee leaves the organization. Because the outcome variable **leave** is binary, the modeling task is a **classification problem**.

Both **parametric** and **non-parametric** models are developed to compare different modeling approaches. Parametric models assume a specific functional form between predictors and the outcome, while non-parametric models are more flexible and can capture complex nonlinear relationships.

The following models are built:

Parametric models:
- Logistic Regression
- Logistic Regression with LASSO regularization
- Generalized Additive Model (GAM)

Non-parametric models:
- Random Forest (tree-based ensemble model)
- Support Vector Machine (SVM)

Deep learning model:
- Neural network with one hidden layer

In addition, the potential usefulness of **Principal Component Analysis (PCA)** and **cluster analysis** for dimensionality reduction and feature augmentation is discussed.

In [ ]:
from sklearn.model_selection import train_test_split

# Separate predictors and outcome
X = df.drop(columns=["leave"])
y = df["leave"].astype(int)

# Convert categorical variables to dummy variables
X = pd.get_dummies(X, drop_first=True, dtype=float)

# Stratification preserves the turnover rate in both partitions
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).sort_index().round(3))
print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True).sort_index().round(3))


### 3.1 Logistic Regression

Logistic regression is a parametric model used for binary classification problems. It models the probability that an observation belongs to a particular class using the logistic function.

This model serves as a baseline model because it provides interpretable coefficients that indicate how each predictor affects the likelihood of employee turnover.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Scaling fixes convergence problems; balanced weights address class imbalance
log_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            class_weight="balanced",
            random_state=42
        )
    )
])

log_model.fit(X_train, y_train)

log_train_prob = log_model.predict_proba(X_train)[:, 1]
log_test_prob = log_model.predict_proba(X_test)[:, 1]
log_train_pred = (log_train_prob >= 0.50).astype(int)
log_test_pred = (log_test_prob >= 0.50).astype(int)

print("Balanced Logistic Regression trained successfully.")


### 3.2 Logistic Regression with LASSO

LASSO (Least Absolute Shrinkage and Selection Operator) introduces a penalty term that shrinks some regression coefficients toward zero. This helps reduce overfitting and can perform variable selection by removing less important predictors.

This approach is particularly useful when the number of predictors is relatively large or when multicollinearity exists among predictors.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

lasso_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegressionCV(
            penalty="l1",
            solver="saga",
            cv=5,
            scoring="roc_auc",
            class_weight="balanced",
            max_iter=5000,
            n_jobs=-1,
            random_state=42
        )
    )
])

lasso_model.fit(X_train, y_train)

lasso_train_prob = lasso_model.predict_proba(X_train)[:, 1]
lasso_test_prob = lasso_model.predict_proba(X_test)[:, 1]
lasso_train_pred = (lasso_train_prob >= 0.50).astype(int)
lasso_test_pred = (lasso_test_prob >= 0.50).astype(int)

print("Balanced LASSO Logistic Regression trained successfully.")


### 3.3 Generalized Additive Model (GAM)

Generalized Additive Models extend generalized linear models by allowing nonlinear relationships between predictors and the outcome variable through smoothing functions.

Bivariate plots between predictors and the outcome can reveal nonlinear patterns. When such patterns are present, GAM models can capture these relationships using smooth functions rather than assuming linear effects.

In [ ]:
!pip install pygam

In [ ]:
from pygam import LogisticGAM, s
from sklearn.preprocessing import StandardScaler
import numpy as np

# Use only continuous predictors for GAM
continuous_vars = [
    "Salary",
    "JobInsecurity",
    "JobSatisfaction",
    "Tenure",
    "EmotionalExhaustion",
    "SupervisorSatisfaction",
    "CareerAdvancement",
    "WorkLifeBalance",
    "OrgAlignment",
    "LaborMarketCondition",
    "Salary_JS",
    "JS_SupS",
    "Tenure_CA",
    "Exhaustion_WLB",
    "CA_Align"
]

X_train_gam = X_train[continuous_vars]
X_test_gam = X_test[continuous_vars]

# Scale continuous predictors
scaler_gam = StandardScaler()
X_train_gam_scaled = scaler_gam.fit_transform(X_train_gam)
X_test_gam_scaled = scaler_gam.transform(X_test_gam)

# Specify smooth terms and tune lambda
gam = LogisticGAM(
    s(0) + s(1) + s(2) + s(3) + s(4) +
    s(5) + s(6) + s(7) + s(8) + s(9) +
    s(10) + s(11) + s(12) + s(13) + s(14)
)

gam.gridsearch(
    X_train_gam_scaled,
    y_train,
    lam=np.logspace(-2, 2, 5)
)

# Predictions
gam_train_prob = gam.predict_proba(X_train_gam_scaled)
gam_test_prob = gam.predict_proba(X_test_gam_scaled)

gam_train_pred = (gam_train_prob >= 0.5).astype(int)
gam_test_pred = (gam_test_prob >= 0.5).astype(int)

### 3.3.1 Generalized Additive Model (GAM)

The initial GAM specification produced convergence problems when applied to the full predictor matrix. This likely occurred because the one-hot encoded design matrix included many dummy variables and interaction-related predictors, which made estimation unstable. To address this, the GAM was fit using the main continuous predictors only, after standardization, and the smoothing penalty was tuned using grid search. This approach is appropriate because GAMs are primarily intended to capture nonlinear effects of continuous predictors.

### 3.4 Random Forest

Random Forest is a tree-based ensemble model that constructs many decision trees and combines their predictions. This method can automatically capture nonlinear relationships and interactions among predictors.

Random Forest was selected as the tree-based ensemble method because it is relatively stable, handles nonlinearities well, and generally requires less hyperparameter tuning than gradient boosting methods.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Restrict tree complexity to reduce overfitting
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_train_prob = rf_model.predict_proba(X_train)[:, 1]
rf_test_prob = rf_model.predict_proba(X_test)[:, 1]
rf_train_pred = (rf_train_prob >= 0.50).astype(int)
rf_test_pred = (rf_test_prob >= 0.50).astype(int)

print("Balanced Random Forest trained successfully.")


### 3.5 Support Vector Machine (SVM)

Support Vector Machines classify observations by identifying the optimal decision boundary that separates classes.

Because relationships between predictors and employee turnover may be nonlinear, a **radial basis function (RBF) kernel** is used. The RBF kernel allows the model to capture nonlinear decision boundaries in the feature space.

In [ ]:
from sklearn.svm import SVC

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "svm",
        SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42
        )
    )
])

svm_model.fit(X_train, y_train)

svm_train_prob = svm_model.predict_proba(X_train)[:, 1]
svm_test_prob = svm_model.predict_proba(X_test)[:, 1]
svm_train_pred = (svm_train_prob >= 0.50).astype(int)
svm_test_pred = (svm_test_prob >= 0.50).astype(int)

print("Balanced SVM trained successfully.")


### 3.6 Deep Learning Model

A neural network model is implemented with the following architecture:

- one hidden layer
- number of hidden nodes equal to **twice the number of input predictors**
- ReLU activation in the hidden layer
- sigmoid activation in the output layer because the outcome variable is binary

Neural networks are flexible models capable of capturing complex nonlinear patterns between predictors and the outcome.

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

tf.keras.utils.set_random_seed(42)

# Neural networks require standardized numerical inputs
scaler_nn = StandardScaler()
X_train_nn = scaler_nn.fit_transform(X_train).astype("float32")
X_test_nn = scaler_nn.transform(X_test).astype("float32")

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
nn_class_weights = dict(zip(classes, weights))

num_predictors = X_train_nn.shape[1]

nn_model = Sequential([
    Input(shape=(num_predictors,)),
    Dense(2 * num_predictors, activation="relu"),
    Dropout(0.20),
    Dense(1, activation="sigmoid")
])

nn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

early_stopping = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=5,
    restore_best_weights=True
)

history = nn_model.fit(
    X_train_nn,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.20,
    class_weight=nn_class_weights,
    callbacks=[early_stopping],
    verbose=0
)

nn_train_prob = nn_model.predict(X_train_nn, verbose=0).ravel()
nn_test_prob = nn_model.predict(X_test_nn, verbose=0).ravel()
nn_train_pred = (nn_train_prob >= 0.50).astype(int)
nn_test_pred = (nn_test_prob >= 0.50).astype(int)

print("Class-balanced neural network trained successfully.")


### Principal Component Analysis (PCA)

Principal Component Analysis is a dimensionality-reduction technique that transforms correlated predictors into a smaller set of uncorrelated components. PCA can be useful when predictors are highly correlated or when the number of predictors is large relative to the number of observations.

In this dataset, PCA could potentially reduce multicollinearity among predictors such as the interaction variables. However, PCA also reduces interpretability because the resulting components are linear combinations of the original variables. Since interpretability is valuable for understanding the drivers of employee turnover, PCA was not implemented in this analysis. If PCA were applied, the principal components could be used as inputs for models such as logistic regression, SVM, or neural networks.

### Cluster Analysis

Cluster analysis is an unsupervised learning technique that groups observations based on similarities among predictors. Applying cluster analysis to employee characteristics could reveal latent groups of employees with similar workplace experiences or job conditions.

If meaningful clusters were identified, cluster membership could be added as an additional predictor in supervised learning models. This could improve predictive performance by capturing group-level patterns in employee behavior. However, because the primary goal of this analysis is predictive modeling of turnover, cluster analysis was not implemented directly in this study.

# 5. Reporting Results

Because employee turnover is imbalanced, accuracy alone is misleading. The
models are evaluated using **balanced accuracy, precision, recall, F1-score,
ROC-AUC, and the confusion matrix**. Recall is especially important because it
measures how many employees who actually leave are identified by the model.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
import pandas as pd

def classification_metrics(y_true, predictions, probabilities):
    tn, fp, fn, tp = confusion_matrix(y_true, predictions).ravel()

    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1-Score": f1_score(
            y_true, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "True Positives": tp,
        "True Negatives": tn,
        "False Positives": fp,
        "False Negatives": fn
    }


In [ ]:
model_outputs = {
    "Logistic Regression": (
        log_test_pred,
        log_test_prob
    ),
    "LASSO Logistic Regression": (
        lasso_test_pred,
        lasso_test_prob
    ),
    "GAM": (
        gam_test_pred,
        gam_test_prob
    ),
    "Random Forest": (
        rf_test_pred,
        rf_test_prob
    ),
    "SVM": (
        svm_test_pred,
        svm_test_prob
    ),
    "Neural Network": (
        nn_test_pred,
        nn_test_prob
    )
}


In [ ]:
results = {}

for model_name, (predictions, probabilities) in model_outputs.items():
    results[model_name] = classification_metrics(
        y_test,
        predictions,
        probabilities
    )


In [ ]:
results_table = (
    pd.DataFrame(results)
    .T
    .sort_values(
        by=["ROC-AUC", "Recall"],
        ascending=False
    )
)

metric_columns = [
    "Accuracy",
    "Balanced Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "ROC-AUC"
]

results_table[metric_columns] = results_table[metric_columns].round(3)
display(results_table)


In [ ]:
import pandas as pd

importance = rf_model.feature_importances_

importance_df = pd.DataFrame({
"Variable": X_train.columns,
"Importance": importance
})

importance_df = importance_df.sort_values(
by="Importance",
ascending=False
)

importance_df["Rank"] = range(1, len(importance_df)+1)

importance_df.head(15)

### Variable Importance

Variable importance was estimated using the Random Forest model. Random Forest calculates feature importance by measuring how much each predictor contributes to reducing classification error across the ensemble of decision trees.

Variables with higher importance scores contribute more strongly to predicting employee turnover.

In [ ]:
importance_table = importance_df[["Variable","Rank"]]

importance_table

In [ ]:
# Display a confusion matrix for every model
confusion_tables = {}

for model_name, (predictions, _) in model_outputs.items():
    cm = confusion_matrix(y_test, predictions)
    confusion_tables[model_name] = pd.DataFrame(
        cm,
        index=["Actual Stay", "Actual Leave"],
        columns=["Predicted Stay", "Predicted Leave"]
    )

for model_name, table in confusion_tables.items():
    print(f"\n{model_name}")
    display(table)


### Interpretation of Model Performance

The corrected evaluation accounts for class imbalance. Accuracy is reported,
but model selection should focus on **ROC-AUC, recall, F1-score, and balanced
accuracy**. A useful turnover model must identify at least some employees who
leave; high accuracy achieved by predicting only the majority class is not
considered successful.

Class weighting gives leaving employees more influence during training.
Standardization improves convergence for Logistic Regression, LASSO, SVM, and
the neural network. The Random Forest is constrained to reduce the severe
training overfitting observed in the original notebook.


In [ ]:
# Rank models by discrimination and churn-detection ability
ranking = results_table[
    [
        "ROC-AUC",
        "Recall",
        "F1-Score",
        "Balanced Accuracy",
        "Accuracy"
    ]
]

display(ranking)


### Model Selection Guidance

The preferred model should have the strongest combination of ROC-AUC, recall,
F1-score, and balanced accuracy. Accuracy should not be used by itself because
most employees remain with the organization. If ROC-AUC remains close to 0.50
after these corrections, the appropriate conclusion is that the available
predictors do not provide enough out-of-sample signal for dependable individual
turnover prediction.


# Data-driven conclusion based on the rerun results
best_model_name = results_table.index[0]
best_result = results_table.iloc[0]

print("--- Final Model Assessment ---")
print("Highest-ranked model:", best_model_name)
print(f"ROC-AUC: {best_result['ROC-AUC']:.3f}")
print(f"Recall: {best_result['Recall']:.3f}")
print(f"F1-score: {best_result['F1-Score']:.3f}")
print(
    f"Balanced accuracy: "
    f"{best_result['Balanced Accuracy']:.3f}"
)

if best_result["ROC-AUC"] < 0.60:
    print(
        "\nConclusion: Even after imbalance-aware training, the models "
        "have limited discriminatory power. The project demonstrates a "
        "valid modeling workflow, but the available data should not be "
        "used for dependable individual turnover decisions without "
        "additional predictive variables and validation."
    )
else:
    print(
        "\nConclusion: The highest-ranked model shows useful predictive "
        "signal. Before deployment, it should still be validated on new "
        "data and reviewed for fairness across employee groups."
    )
